# Vectorless RAG with PageIndex

This notebook builds a model-agnostic, vectorless RAG workflow over PDF files stored in a local `data/` folder.

The flow is:

1. Install and configure dependencies.
2. Load a PDF from `data/` and submit it to PageIndex.
3. Use an LLM to reason over the PageIndex tree and select relevant nodes.
4. Extract the evidence text from those nodes.
5. Ask the LLM for a final answer plus a concise explainability summary.

The notebook is intentionally modular so each step can be inspected or replaced independently.

## 1. Prerequisites

This notebook assumes you have:

- A PDF document in a local `data/` folder.
- A PageIndex API key for tree generation.
- A Groq API key for the free LLM endpoint.

The LLM layer uses [Groq](https://groq.com/) via its OpenAI-compatible endpoint. You can choose between `llama-3.3-70b-versatile` (larger) and `llama-3.1-8b-instant` (faster) at runtime.

### API Key Setup

You will need two API keys:

| Key | Where to get it |
|-----|------------------|
| `GROQ_API_KEY` | [https://console.groq.com/keys](https://console.groq.com/keys) |
| `PAGEINDEX_API_KEY` | [https://www.pageindex.ai](https://www.pageindex.ai) |

Set them as environment variables or in a `.env` file in the same directory as this notebook:

```
GROQ_API_KEY=gsk_...
PAGEINDEX_API_KEY=...
```

In [ ]:
%pip install -q --upgrade pageindex openai python-dotenv pymupdf

## 2. Configuration

The notebook reads environment variables from a `.env` file or your shell:

- `GROQ_API_KEY`: Groq API key (from [console.groq.com/keys](https://console.groq.com/keys)).
- `PAGEINDEX_API_KEY`: PageIndex API key.
- `PDF_NAME`: Optional filename in `data/` if you want to pin a specific PDF.

You will be prompted to choose between two Groq models at runtime:
- `llama-3.3-70b-versatile` — larger, more capable.
- `llama-3.1-8b-instant` — faster, lighter.

Groq provides a free OpenAI-compatible endpoint, so no `LLM_BASE_URL` is needed.

In [ ]:
from pathlib import Path
import json
import os
import re
import time
from typing import Any

from dotenv import load_dotenv
from openai import AsyncOpenAI
from pageindex import PageIndexClient
import pageindex.utils as pi_utils

load_dotenv()

DATA_DIR = Path("data")
CACHE_DIR = DATA_DIR / "cache"
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY", "").strip()
LLM_API_KEY = os.getenv("GROQ_API_KEY", "").strip()
LLM_BASE_URL = "https://api.groq.com/openai/v1"
PDF_NAME = os.getenv("PDF_NAME")

MODELS = {
    "1": ("llama-3.3-70b-versatile", "Llama 3.3 70B — larger, more capable"),
    "2": ("llama-3.1-8b-instant", "Llama 3.1 8B — faster, lighter"),
}

print("Choose an LLM model:")
for key, (name, desc) in MODELS.items():
    print(f"  {key}. {name} — {desc}")

choice = input("Enter 1 or 2: ").strip()
LLM_MODEL = MODELS.get(choice, MODELS["1"])[0]
print(f"Using model: {LLM_MODEL}")

if not PAGEINDEX_API_KEY:
    print("Set PAGEINDEX_API_KEY before running the PageIndex cells.")
if not LLM_API_KEY:
    print("Set GROQ_API_KEY before running the retrieval and answer cells.")

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY) if PAGEINDEX_API_KEY else None
llm_client = AsyncOpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL) if LLM_API_KEY else None


def extract_json(text: str) -> dict[str, Any]:
    match = re.search(r"\{.*\}", text, re.S)
    if not match:
        raise ValueError(f"Expected JSON output, got: {text[:500]}")
    return json.loads(match.group(0))


async def call_llm(system_prompt: str, user_prompt: str, model: str = LLM_MODEL, temperature: float = 0.0) -> str:
    if llm_client is None:
        raise RuntimeError("GROQ_API_KEY is not configured.")
    response = await llm_client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content.strip()


def preview_text(text: str, limit: int = 1200) -> str:
    text = text.strip()
    return text if len(text) <= limit else text[:limit].rstrip() + "..."

## 3. Load a PDF from `data/`

This notebook works with one user-provided PDF at a time. If `PDF_NAME` is set, the notebook uses that file from `data/`; otherwise it uses the first PDF it finds.

The PageIndex step below creates the tree structure for this document without using embeddings or a vector database.

In [ ]:
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Missing data folder: {DATA_DIR.resolve()}")

pdf_candidates = sorted(DATA_DIR.glob("*.pdf"))
if not pdf_candidates:
    raise FileNotFoundError(f"No PDF files found in {DATA_DIR.resolve()}")

if PDF_NAME:
    matching = [path for path in pdf_candidates if path.name == PDF_NAME]
    if not matching:
        available = ", ".join(path.name for path in pdf_candidates)
        raise FileNotFoundError(f"PDF_NAME={PDF_NAME!r} was not found in data/. Available files: {available}")
    pdf_path = matching[0]
else:
    pdf_path = pdf_candidates[0]

print(f"Using PDF: {pdf_path}")

## 4. Build the PageIndex tree

This cell submits the selected PDF to PageIndex and retrieves the generated tree.

The tree is cached locally in `data/cache/` so subsequent runs skip the PageIndex API call for the same document.

If the document is still processing, the notebook polls with a progress bar until the tree is ready.

In [ ]:
CACHE_DIR.mkdir(exist_ok=True)
cache_path = CACHE_DIR / f"{pdf_path.stem}_tree.json"

if cache_path.exists():
    print(f"Loading cached tree from {cache_path}")
    tree = json.loads(cache_path.read_text())
else:
    if pi_client is None:
        raise RuntimeError("PAGEINDEX_API_KEY is not configured.")

    submitted = pi_client.submit_document(str(pdf_path))
    doc_id = submitted.get("doc_id") or submitted.get("result", {}).get("doc_id")
    if not doc_id:
        raise RuntimeError(f"Could not read doc_id from PageIndex response: {submitted}")

    print(f"Submitted document id: {doc_id}")
    print("Waiting for PageIndex to process the document...")

    poll_interval = 5
    max_wait = 300
    elapsed = 0
    spinner = ["|", "/", "-", "\\"]
    while elapsed < max_wait:
        if pi_client.is_retrieval_ready(doc_id):
            break
        idx = (elapsed // poll_interval) % len(spinner)
        print(f"\r  {spinner[idx]}  Waiting... {elapsed}s / {max_wait}s", end="", flush=True)
        time.sleep(poll_interval)
        elapsed += poll_interval
    else:
        raise RuntimeError(
            f"PageIndex did not finish within {max_wait}s. Rerun this cell later."
        )
    print(f"\r  Done! Processed in {elapsed}s.{' ' * 20}")

    tree_response = pi_client.get_tree(doc_id, node_summary=True)
    tree = tree_response.get("result", tree_response)

    cache_path.write_text(json.dumps(tree))
    print(f"Tree cached to {cache_path}")

print("Tree ready. Top-level preview:")
pi_utils.print_tree(tree[:2] if isinstance(tree, list) else tree)

## 5. Prepare retrieval helpers

The next cells use the tree structure for two passes:

1. Tree search: the LLM selects the node IDs most likely to contain the answer.
2. Evidence extraction: the notebook pulls the full text for those nodes and passes it into the final answer prompt.

This keeps the workflow vectorless while still giving you transparent, inspectable retrieval decisions.

In [ ]:
node_map = pi_utils.create_node_mapping(tree)

def tree_as_prompt_text(document_tree: object) -> str:
    tree_copy = document_tree.copy() if isinstance(document_tree, dict) else document_tree
    tree_without_text = pi_utils.remove_fields(tree_copy, fields=["text"])
    return json.dumps(tree_without_text, indent=2)


def collect_node_text(node_ids: list[str]) -> str:
    parts: list[str] = []
    for node_id in node_ids:
        node = node_map.get(node_id)
        if not node:
            continue
        text = node.get("text", "")
        title = node.get("title", node_id)
        page_index = node.get("page_index", "?")
        parts.append(f"[node={node_id} page={page_index} title={title}]\n{text}")
    return "\n\n".join(parts)


print(f"Indexed nodes: {len(node_map)}")

## 6. Reasoning-based tree search

This is the vectorless retrieval step.

The LLM receives a question entered by the user and a compact PageIndex tree containing node titles and summaries. It then chooses the node IDs most likely to contain the answer and returns a short retrieval explanation that you can inspect directly.

In [ ]:
QUESTION = input("Enter your question about the PDF: ").strip()
if not QUESTION:
    raise ValueError("A question is required to continue.")

retrieval_system_prompt = """
You are a document retrieval assistant for a vectorless RAG system.

You will receive a user question and a PageIndex tree made of node titles and summaries.
Select only the node IDs that are likely to contain answer-bearing evidence.

Return valid JSON with this shape:
{
  "thinking": "short, evidence-based retrieval explanation",
  "node_list": ["node_id_1", "node_id_2"]
}

Do not output markdown, prose, or extra keys.
Keep the explanation brief and grounded in the tree.
""".strip()

retrieval_user_prompt = f"""
Question:
{QUESTION}

PageIndex tree:
{tree_as_prompt_text(tree)}
""".strip()

retrieval_response = await call_llm(retrieval_system_prompt, retrieval_user_prompt)
retrieval_json = extract_json(retrieval_response)
selected_node_ids = retrieval_json.get("node_list", [])

print("Selected node IDs:", selected_node_ids)
print("Retrieval explainability:")
print(retrieval_json.get("thinking", ""))

## 7. Extract evidence text

The selected node IDs are converted back into their source text. That evidence is what the final answer step is allowed to use.

Because the notebook passes the raw node text forward, you can inspect exactly which passages influenced the result.

In [ ]:
evidence_text = collect_node_text(selected_node_ids)
if not evidence_text.strip():
    raise RuntimeError("No evidence text was collected. Check the selected node IDs and tree mapping.")

print("Evidence preview:\n")
print(preview_text(evidence_text, limit=3000))

## 8. Final answer and explainability

The final LLM call uses only the retrieved evidence text and the original question.

To keep the workflow auditable, the response is structured as JSON with two top-level parts:

- `final_answer`: the user-facing answer.
- `explainability`: a short summary of why the answer was produced and which evidence was used.

In [ ]:
final_system_prompt = """
You are a careful assistant answering questions about a PDF document.

Use only the evidence text provided by the notebook.
Do not invent facts or rely on outside knowledge.
Return valid JSON with this shape:
{
  "final_answer": "concise answer for the user",
  "explainability": {
    "retrieval_summary": "brief note about why the selected evidence is relevant",
    "evidence_used": ["short evidence note 1", "short evidence note 2"]
  }
}

Keep the explanation short and grounded in the provided context.
""".strip()

final_user_prompt = f"""
Question:
{QUESTION}

Evidence context:
{evidence_text}
""".strip()

final_response = await call_llm(final_system_prompt, final_user_prompt)
final_json = extract_json(final_response)

print("Final answer:\n")
print(final_json.get("final_answer", ""))

print("\nExplainability:\n")
print(json.dumps(final_json.get("explainability", {}), indent=2))